# Heart Disease Prediction — ML Model Export

Dataset: UCI Heart Disease (303 rows, 13 features, binary target)

Pipeline: EDA → Preprocessing → RandomForest → ONNX export

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for script execution
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
)

sns.set_theme(style='whitegrid')
print('Libraries loaded')

Libraries loaded


## 1. Load Data

In [2]:
df = pd.read_csv('data/heart.csv')
print('Shape:', df.shape)
df.head()

Shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


## 2. EDA — Exploratory Data Analysis

In [3]:
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Descriptive Statistics ===')
df.describe()

=== Data Types ===
age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca            int64
thal          int64
target        int64
dtype: object

=== Descriptive Statistics ===


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000
mean,54.366337,0.683168,0.966997,131.623762,246.264026,0.148515,0.528053,149.646865,0.326733,1.039604,1.399340,0.729373,2.313531,0.544554
std,9.082101,0.466011,1.032052,17.538143,51.830751,0.356198,0.525860,22.905161,0.469794,1.161075,0.616226,1.022606,0.612277,0.498835
min,29.000000,0.000000,0.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,47.500000,0.000000,0.000000,120.000000,211.000000,0.000000,0.000000,133.500000,0.000000,0.000000,1.000000,0.000000,2.000000,0.000000
50%,55.000000,1.000000,1.000000,130.000000,240.000000,0.000000,1.000000,153.000000,0.000000,0.800000,1.000000,0.000000,2.000000,1.000000
75%,61.000000,1.000000,2.000000,140.000000,274.500000,0.000000,1.000000,166.000000,1.000000,1.600000,2.000000,1.000000,3.000000,1.000000
max,77.000000,1.000000,3.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,2.000000,4.000000,3.000000,1.000000


In [4]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing: {missing.sum()}')

=== Missing Values ===
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

Total missing: 0


In [5]:
# Class balance
fig, ax = plt.subplots(figsize=(5, 4))
counts = df['target'].value_counts().sort_index()
ax.bar(['No Disease (0)', 'Disease (1)'], counts.values, color=['steelblue', 'tomato'])
ax.set_title('Class Balance')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 2, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()
print(f'\nClass distribution:\n{counts}')


Class distribution:
target
0    138
1    165
Name: count, dtype: int64


/tmp/ipykernel_36593/736681088.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 9))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

/tmp/ipykernel_36593/257016817.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Numerical feature distributions by target
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    for target_val, color, label in [(0, 'steelblue', 'No Disease'), (1, 'tomato', 'Disease')]:
        axes[i].hist(df[df['target'] == target_val][col],
                     alpha=0.6, color=color, label=label, bins=20)
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

axes[-1].axis('off')
fig.suptitle('Numerical Feature Distributions by Target', fontsize=14)
plt.tight_layout()
plt.show()

/tmp/ipykernel_36593/3238909179.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Categorical feature counts by target
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    ct = df.groupby([col, 'target']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=axes[i], color=['steelblue', 'tomato'], alpha=0.8)
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=0)
    axes[i].legend(['No Disease', 'Disease'])

fig.suptitle('Categorical Features by Target', fontsize=14)
plt.tight_layout()
plt.show()

/tmp/ipykernel_36593/2786251273.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Preprocessing Pipeline

We use **integer column indices** (not string names) so that `skl2onnx` can export the pipeline to a single `float_input` tensor without column-name resolution errors.

In [9]:
# Fixed column order — must match ToFeatureArray() in PatientInput.cs
ALL_FEATURES = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak',
                'sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

# Integer indices for ColumnTransformer (required for skl2onnx single-tensor export)
NUMERICAL_IDX   = list(range(5))       # indices 0-4  : age, trestbps, chol, thalach, oldpeak
CATEGORICAL_IDX = list(range(5, 13))   # indices 5-12 : sex, cp, fbs, restecg, exang, slope, ca, thal

# Convert to numpy float32 — training on arrays avoids column-name issues
X = df[ALL_FEATURES].values.astype(np.float32)
y = df['target'].values

print('Feature matrix shape:', X.shape)
print('Target shape:', y.shape)
print('\nFeature order (must match C# ToFeatureArray):')
for i, f in enumerate(ALL_FEATURES):
    print(f'  [{i:2d}] {f}')

Feature matrix shape: (303, 13)
Target shape: (303,)

Feature order (must match C# ToFeatureArray):
  [ 0] age
  [ 1] trestbps
  [ 2] chol
  [ 3] thalach
  [ 4] oldpeak
  [ 5] sex
  [ 6] cp
  [ 7] fbs
  [ 8] restecg
  [ 9] exang
  [10] slope
  [11] ca
  [12] thal


In [10]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUMERICAL_IDX),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_IDX),
])

pipeline = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42)),
])

print('Pipeline defined:')
print(pipeline)

Pipeline defined:
Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  [0, 1, 2, 3, 4]),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [5, 6, 7, 8, 9, 10, 11,
                                                   12])])),
                ('clf', RandomForestClassifier(random_state=42))])


## 4. Train / Test Split & Model Training

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

pipeline.fit(X_train, y_train)
print('\nModel trained successfully!')

Train: (242, 13), Test: (61, 13)



Model trained successfully!


## 5. Evaluation

In [12]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print('=== Classification Metrics ===')
print(f'Accuracy : {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'F1 Score : {f1:.4f}')

=== Classification Metrics ===
Accuracy : 0.8197
Precision: 0.7750
Recall   : 0.9394
F1 Score : 0.8493


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Disease', 'Disease'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='tomato', lw=2, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.02])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

/tmp/ipykernel_36593/3370835536.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. ONNX Export

In [14]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import os

# Single float tensor input with 13 features
# Works because ColumnTransformer uses integer indices, not column names
initial_type = [('float_input', FloatTensorType([None, 13]))]
onnx_model = convert_sklearn(pipeline, initial_types=initial_type, target_opset=17)

os.makedirs('../model', exist_ok=True)
onnx_path = '../model/heart_disease.onnx'

with open(onnx_path, 'wb') as f:
    f.write(onnx_model.SerializeToString())

size_kb = os.path.getsize(onnx_path) / 1024
print(f'ONNX model saved to: {onnx_path}')
print(f'File size: {size_kb:.1f} KB')

ONNX model saved to: ../model/heart_disease.onnx
File size: 331.3 KB


## 7. ONNX Sanity Check

Load the exported `.onnx` with `onnxruntime` and verify predictions match the sklearn pipeline.

In [15]:
import onnxruntime as rt

sess = rt.InferenceSession(onnx_path)
input_name = sess.get_inputs()[0].name
print('ONNX input name :', input_name)
print('ONNX input shape:', sess.get_inputs()[0].shape)
print('ONNX outputs    :', [o.name for o in sess.get_outputs()])

ONNX input name : float_input
ONNX input shape: [None, 13]
ONNX outputs    : ['output_label', 'output_probability']


In [16]:
# Run 5 test-set samples through ONNX and compare with sklearn
sample = X_test[:5].astype(np.float32)

onnx_out    = sess.run(None, {input_name: sample})
onnx_labels = onnx_out[0]                                    # predicted class
onnx_probs  = np.array([list(p.values())[1] for p in onnx_out[1]])  # P(class=1)

sk_labels = pipeline.predict(X_test[:5])
sk_probs  = pipeline.predict_proba(X_test[:5])[:, 1]

comparison = pd.DataFrame({
    'sklearn_label': sk_labels,
    'onnx_label'  : onnx_labels,
    'sklearn_prob': sk_probs.round(4),
    'onnx_prob'   : onnx_probs.round(4),
    'match'       : sk_labels == onnx_labels,
})
print(comparison.to_string())
print(f'\nAll predictions match: {comparison["match"].all()}')

   sklearn_label  onnx_label  sklearn_prob  onnx_prob  match
0              0           0          0.08       0.08   True
1              0           0          0.44       0.44   True
2              0           0          0.02       0.02   True
3              1           1          0.68       0.69   True
4              1           1          0.54       0.53   True

All predictions match: True


In [17]:
# Print a known test row for Blazor app verification
print('=== Sample input for Blazor app verification ===')
print('Feature order:', ALL_FEATURES)
print('Values       :', X_test[0].tolist())
print(f'\nExpected label    : {y_test[0]}')
print(f'sklearn probability: {sk_probs[0]:.4f}')

=== Sample input for Blazor app verification ===
Feature order: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
Values       : [57.0, 150.0, 276.0, 112.0, 0.6000000238418579, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0]

Expected label    : 0
sklearn probability: 0.0800
